In [1]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import ParameterGrid
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_curve,
)

SEED = 123


def clean_seq(s: str) -> str:
    s = str(s).upper()
    out = []
    for c in s:
        out.append(c if c in "ACGT" else "N")
    return "".join(out)


def load_df(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    if "sequence" not in df.columns or "label" not in df.columns:
        raise ValueError(f"{path} must contain columns: sequence, label")
    df["sequence"] = df["sequence"].astype(str).map(clean_seq)
    df["label"] = df["label"].astype(int)
    return df


def pick_best_threshold_f1(y_true, p_prob, grid=None):
    if grid is None:
        grid = np.linspace(0, 1, 101)

    best_t, best_f1 = 0.5, -1.0
    y_true = np.asarray(y_true)

    for t in grid:
        y_pred = (p_prob >= t).astype(int)
        tp = np.sum((y_true == 1) & (y_pred == 1))
        fp = np.sum((y_true == 0) & (y_pred == 1))
        fn = np.sum((y_true == 1) & (y_pred == 0))

        prec = tp / (tp + fp + 1e-12)
        rec = tp / (tp + fn + 1e-12)
        f1 = 2 * prec * rec / (prec + rec + 1e-12)

        if f1 > best_f1:
            best_f1, best_t = float(f1), float(t)

    return best_t, best_f1


def save_roc(y_true, p_prob, out_dir, prefix="hold"):
    os.makedirs(out_dir, exist_ok=True)

    fpr, tpr, thresholds = roc_curve(y_true, p_prob)
    auc = roc_auc_score(y_true, p_prob) if len(np.unique(y_true)) > 1 else float("nan")

    roc_df = pd.DataFrame({"fpr": fpr, "tpr": tpr, "threshold": thresholds})
    roc_csv = os.path.join(out_dir, f"{prefix}_roc_data.csv")
    roc_df.to_csv(roc_csv, index=False)

    plt.figure()
    plt.plot(fpr, tpr, label=f"ROC (AUC={auc:.4f})")
    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"{prefix.upper()} ROC Curve")
    plt.legend(loc="lower right")
    roc_png = os.path.join(out_dir, f"{prefix}_roc_curve.png")
    plt.savefig(roc_png, dpi=200, bbox_inches="tight")
    plt.close()

    return float(auc), roc_csv, roc_png


def fit_best_model_on_train_val(train_df, val_df, param_grid, seed=SEED):
    """
    Model selection on validation set:
    1) prioritize AUC
    2) tie-break by AP
    Threshold is then optimized on validation set by max F1.
    """
    X_train, y_train = train_df["sequence"], train_df["label"].values
    X_val, y_val = val_df["sequence"], val_df["label"].values

    best = {"ap": -1, "auc": -1, "params": None, "thresh": 0.5, "model": None}

    for params in ParameterGrid(param_grid):
        clf = Pipeline(
            [
                (
                    "vec",
                    CountVectorizer(
                        analyzer="char",
                        ngram_range=(params["K"], params["K"]),
                        lowercase=False,
                        min_df=params["min_df"],
                    ),
                ),
                (
                    "lr",
                    LogisticRegression(
                        max_iter=2000,
                        class_weight="balanced",
                        solver="liblinear",
                        C=params["C"],
                        random_state=seed,
                    ),
                ),
            ]
        )

        clf.fit(X_train, y_train)
        p_val = clf.predict_proba(X_val)[:, 1]

        auc = roc_auc_score(y_val, p_val) if len(np.unique(y_val)) > 1 else float("nan")
        ap = average_precision_score(y_val, p_val)

        t_best, best_f1 = pick_best_threshold_f1(y_val, p_val)

        # 改为：AUC 优先，AP 作为次级排序标准
        better = (auc > best["auc"]) or (auc == best["auc"] and ap > best["ap"])
        if better:
            best.update(
                {
                    "ap": float(ap),
                    "auc": float(auc),
                    "params": params,
                    "thresh": float(t_best),
                    "best_val_f1": float(best_f1),
                    "model": clf,
                }
            )

    return best


def eval_and_save_hold(best, hold_df, out_dir):
    X_hold, y_hold = hold_df["sequence"], hold_df["label"].values
    model = best["model"]
    thresh = best["thresh"]

    p_hold = model.predict_proba(X_hold)[:, 1]
    hold_auc = roc_auc_score(y_hold, p_hold) if len(np.unique(y_hold)) > 1 else float("nan")
    hold_ap = average_precision_score(y_hold, p_hold)

    y_pred = (p_hold >= thresh).astype(int)
    hold_acc = accuracy_score(y_hold, y_pred)
    hold_cm = confusion_matrix(y_hold, y_pred)
    hold_report = classification_report(y_hold, y_pred, digits=4)

    _, roc_csv, roc_png = save_roc(y_hold, p_hold, out_dir, prefix="hold")

    cm_df = pd.DataFrame(hold_cm, index=["true_0", "true_1"], columns=["pred_0", "pred_1"])
    cm_csv = os.path.join(out_dir, "hold_confusion_matrix.csv")
    cm_df.to_csv(cm_csv, index=True)

    summary = {
        "best_params_from_val": best["params"],
        "val_selection_rule": {
            "primary_metric": "AUC",
            "secondary_metric": "AP_if_AUC_tied"
        },
        "val_metrics_for_best_params": {
            "auc": float(best["auc"]),
            "ap": float(best["ap"]),
            "best_f1": float(best.get("best_val_f1", float("nan")))
        },
        "threshold_from_val_f1": float(thresh),
        "threshold_search_grid": {
            "start": 0,
            "end":   1,
            "step": 0.01
        },
        "hold_metrics": {
            "auc": float(hold_auc),
            "ap": float(hold_ap),
            "acc": float(hold_acc)
        },
        "confusion_matrix": hold_cm.tolist(),
        "artifacts": {
            "hold_roc_curve_png": roc_png,
            "hold_roc_data_csv": roc_csv,
            "hold_confusion_matrix_csv": cm_csv,
        },
    }

    with open(os.path.join(out_dir, "hold_summary.json"), "w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)

    with open(os.path.join(out_dir, "hold_report.txt"), "w", encoding="utf-8") as f:
        f.write(f"Best params (chosen on VAL): {best['params']}\n")
        f.write("Model selection rule: primary=AUC, secondary=AP when AUC is tied\n")
        f.write(f"VAL metrics for best params: AUC={best['auc']:.6f} AP={best['ap']:.6f}\n")
        f.write(f"Threshold search grid: 0.0 to 1 with step 0.01\n")
        f.write(f"Threshold (chosen on VAL by max F1): {thresh:.4f}\n\n")
        f.write(f"HOLD AUC: {hold_auc:.6f}\nHOLD AP: {hold_ap:.6f}\nHOLD ACC: {hold_acc:.6f}\n\n")
        f.write("HOLD Confusion Matrix:\n")
        f.write(np.array2string(hold_cm) + "\n\n")
        f.write(hold_report + "\n")
        f.write(f"\nROC PNG: {roc_png}\nROC CSV: {roc_csv}\nCM  CSV: {cm_csv}\n")

    return {
        "auc": float(hold_auc),
        "ap": float(hold_ap),
        "acc": float(hold_acc),
        "thresh": float(thresh),
        "best_params": best["params"],
        "val_auc": float(best["auc"]),
        "val_ap": float(best["ap"]),
        "val_best_f1": float(best.get("best_val_f1", float("nan"))),
        "n_hold": int(len(hold_df)),
        "pos_rate_hold": float(np.mean(y_hold == 1)),
    }


def summarize_rocs(out_root: str, n_reps: int = 10):
    print("\n==================== ROC SUMMARY ====================")

    roc_rows = []
    auc_rows = []

    for rep_i in range(1, n_reps + 1):
        roc_path = os.path.join(out_root, f"rep{rep_i}", "hold_roc_data.csv")
        summary_json = os.path.join(out_root, f"rep{rep_i}", "hold_summary.json")

        if not os.path.exists(roc_path):
            continue

        roc_df = pd.read_csv(roc_path)
        roc_df["rep"] = rep_i
        roc_rows.append(roc_df)

        if os.path.exists(summary_json):
            with open(summary_json, "r", encoding="utf-8") as f:
                data = json.load(f)
            auc_rows.append({"rep": rep_i, "hold_auc": data["hold_metrics"]["auc"]})

    if roc_rows:
        roc_all = pd.concat(roc_rows, ignore_index=True)
        roc_all_csv = os.path.join(out_root, "all_reps_roc_summary.csv")
        roc_all.to_csv(roc_all_csv, index=False)
        print("Saved:", roc_all_csv)

        mean_fpr = np.linspace(0, 1, 200)
        tprs = []

        for rep_i in sorted(roc_all["rep"].unique()):
            df_rep = roc_all[roc_all["rep"] == rep_i].sort_values("fpr")
            fpr = df_rep["fpr"].values
            tpr = df_rep["tpr"].values
            interp_tpr = np.interp(mean_fpr, fpr, tpr)
            interp_tpr[0] = 0.0
            tprs.append(interp_tpr)

        mean_tpr = np.mean(tprs, axis=0)
        std_tpr = np.std(tprs, axis=0)
        mean_auc = np.trapz(mean_tpr, mean_fpr)

        mean_roc_df = pd.DataFrame({"fpr": mean_fpr, "mean_tpr": mean_tpr, "std_tpr": std_tpr})
        mean_roc_csv = os.path.join(out_root, "mean_roc_curve.csv")
        mean_roc_df.to_csv(mean_roc_csv, index=False)
        print("Saved:", mean_roc_csv)
        print(f"Mean AUC (interpolated): {mean_auc:.4f}")

    if auc_rows:
        auc_df = pd.DataFrame(auc_rows).sort_values("rep")
        auc_csv = os.path.join(out_root, "all_reps_auc_summary.csv")
        auc_df.to_csv(auc_csv, index=False)
        print("Saved:", auc_csv)


def main():
    REPS_ROOT = "../../data"
    OUT_ROOT = "./artifacts_by_rep"
    N_REP = 10

    os.makedirs(OUT_ROOT, exist_ok=True)

    param_grid = {
        "K": [4, 5, 6, 7, 8],
        "min_df": [1, 2, 5],
        "C": [0.1, 1.0, 3.0, 10.0],
    }

    rows = []

    for rep_i in range(1, N_REP + 1):
        rep_dir = os.path.join(REPS_ROOT, f"rep{rep_i}")
        train_path = os.path.join(rep_dir, "train.csv")
        val_path = os.path.join(rep_dir, "val.csv")
        hold_path = os.path.join(rep_dir, "hold.csv")

        if not (os.path.exists(train_path) and os.path.exists(val_path) and os.path.exists(hold_path)):
            print(f"[SKIP] rep{rep_i} missing files.")
            continue

        print(f"\n==================== REP {rep_i} ====================")

        train_df = load_df(train_path)
        val_df = load_df(val_path)
        hold_df = load_df(hold_path)

        best = fit_best_model_on_train_val(train_df, val_df, param_grid, seed=SEED)

        rep_out = os.path.join(OUT_ROOT, f"rep{rep_i}")
        os.makedirs(rep_out, exist_ok=True)

        metrics = eval_and_save_hold(best, hold_df, rep_out)

        print("Best params:", metrics["best_params"])
        print(
            f"VAL  AUC={metrics['val_auc']:.4f} | AP={metrics['val_ap']:.4f} "
            f"| bestF1={metrics['val_best_f1']:.4f} | thresh={metrics['thresh']:.3f}"
        )
        print(f"HOLD AUC={metrics['auc']:.4f} | AP={metrics['ap']:.4f} | ACC={metrics['acc']:.4f}")
        print("Saved to:", os.path.abspath(rep_out))

        rows.append({"rep": rep_i, **metrics})

    if rows:
        summary_df = pd.DataFrame(rows).sort_values("rep")
        summary_csv = os.path.join(OUT_ROOT, "all_reps_summary.csv")
        summary_df.to_csv(summary_csv, index=False)
        print("\nSaved summary:", os.path.abspath(summary_csv))

        summarize_rocs(OUT_ROOT, n_reps=N_REP)
    else:
        print("No reps processed.")


if __name__ == "__main__":
    main()



==================== REP 1 ====================
Best params: {'C': 0.1, 'K': 6, 'min_df': 1}
VAL  AUC=0.6867 | AP=0.6768 | bestF1=0.6925 | thresh=0.200
HOLD AUC=0.6992 | AP=0.6795 | ACC=0.6015
Saved to: /home/liusai/26/temp/H3/baseline/LR/artifacts_by_rep/rep1

==================== REP 2 ====================
Best params: {'C': 0.1, 'K': 6, 'min_df': 1}
VAL  AUC=0.6884 | AP=0.6677 | bestF1=0.6903 | thresh=0.180
HOLD AUC=0.7151 | AP=0.7022 | ACC=0.5999
Saved to: /home/liusai/26/temp/H3/baseline/LR/artifacts_by_rep/rep2

==================== REP 3 ====================
Best params: {'C': 0.1, 'K': 5, 'min_df': 1}
VAL  AUC=0.6941 | AP=0.6767 | bestF1=0.6970 | thresh=0.340
HOLD AUC=0.7069 | AP=0.6865 | ACC=0.6234
Saved to: /home/liusai/26/temp/H3/baseline/LR/artifacts_by_rep/rep3

==================== REP 4 ====================
Best params: {'C': 0.1, 'K': 5, 'min_df': 1}
VAL  AUC=0.6947 | AP=0.6784 | bestF1=0.6942 | thresh=0.280
HOLD AUC=0.7144 | AP=0.7017 | ACC=0.6141
Saved to: /home/lius